# <b style='font-size:30px;font-family:Arial'>File-Embedding-Based Collection — Multiple JSON Files (Async)</b>

This notebook demonstrates how to ingest multiple JSON files into a Teradata Vector Store collection using the **Ingestor Pipeline** in **asynchronous (fire-and-forget) mode**.

Calling `.run(sync=False)` returns immediately with a `Task` object — the pipeline continues running in the background on Vantage. This is useful for large datasets where you do not want to block the notebook while ingestion is in progress.

| Step | Description |
|---|---|
| `.files().upsert().run(sync=False)` | Submits the ingestion pipeline and returns a `Task` immediately |
| `task.status()` | Polls the current state of the background task |
| `task.wait()` | Blocks until the task reaches a terminal state (success or failure) |

---
## <b style='font-size:22px;font-family:Arial'>1 · Import Required Libraries</b>

In [1]:
import os
import json
import glob
import getpass

from teradataml import create_context, DataFrame
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema
from teradatagenai import LocalConfig, TeradataAI
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR

---
## <b style='font-size:22px;font-family:Arial'>2 · Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
hostname = getpass.getpass('Enter Teradata Host: ')
username = getpass.getpass('Enter Teradata Username: ')
password = getpass.getpass('Enter Teradata Password: ')

# Create database context
context=create_context(host=hostname, username=username, password=password)
print("✅ Database connection established")
print(f"🌐 Connected to: {hostname}")
print(f"👤 User: {username}")

In [ ]:
from teradataml import set_auth_token

# Refer to help(set_auth_token) for more details on how to set the auth token for your environment.
# Uncomment the below lines if you want to set authentication token using basic authentication.
# You will be prompted to enter the base_url for your environment.

# base_url = getpass.getpass('Enter Base URL: ')
# set_auth_token(base_url= base_url, 
#                username = username, 
#                password = password, 
#                auth_mech = "BASIC")

---
## <b style='font-size:22px;font-family:Arial'>3 · Configure Embedding Model</b>

In [3]:
embedding_model = TeradataAI(
    api_type="td_hosted",
    model_name="amazon.titan-embed-text-v1",
)

---
## <b style='font-size:22px;font-family:Arial'>4 · Discover Files, Configure LocalConfig & ExtractionSchema</b>

The JSON records in this dataset have the structure:
```json
{ "text": "...", "embeddings": [...1536 floats...], "element_id": "...", "metadata": { "filename": "...", "page_number": 1, ... } }
```
The `ExtractionSchema` maps `text` as the data column and `embeddings` as the pre-computed vector column — making this a **FILE_EMBEDDING_BASED** collection (no embedding generation needed at runtime).

In [ ]:
import os, glob
notebook_dir = os.getcwd()
JSON_DIR = os.path.join(os.path.dirname(notebook_dir), "example-data", "json_files", "multiple_json")

json_files = sorted(glob.glob(os.path.join(JSON_DIR, "*.json")))
print(f"Found {len(json_files)} JSON files")

In [ ]:
local_config = LocalConfig(
    files=json_files,
    files_type="json"
)

extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings")
    ],
    metadata_columns=[
        ColumnInfo(name="languages")
    ]
)

Found 25 JSON files


---
## <b style='font-size:22px;font-family:Arial'>Async Ingestor Pipeline</b>

Build the pipeline with `.files()` → `.upsert()` → `.run(sync=False)`.  
The call returns immediately with a `Task` object while ingestion continues in the background on Vantage.

In [6]:
collection_name = "json_multi_file_collection_async"
existing = Collection(name=collection_name)
if existing.exists:
    existing.destroy()

/usr/local/lib/python3.12/dist-packages/teradatagenai/vector_store/collection.py:649: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


In [ ]:
ingestor = (
    Ingestor(
        name=collection_name,
        type=CollectionType.FILE_EMBEDDING_BASED,
        description="Multi-file JSON ingestion — async mode"
    )
    .files(
        files=local_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema
    )
    .upsert(embedding_model=embedding_model)
)
print("Ingestor configured.")

# This call returns immediately — the pipeline runs in the background on Vantage
task = ingestor.run(sync=False)

Ingestor configured.


Initializing collection pipeline (async)...


/usr/local/lib/python3.12/dist-packages/teradatagenai/vector_store/collection.py:649: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Ingest completed successfully
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

#### Monitor Task Status

Use `task.done()` to poll whether the task is complete or not

In [8]:
# Non-blocking: check current state without waiting
print(task.done())

False


In [9]:
# Result after the task is complete
print(task.done())

True


#### Inspect per-file ingestion status

After the task completes, call `ingestor.get_file_metadata()` to see the ingestion status of every file.  
If any file shows a status other than `SUCCESS`, the metadata will tell you what went wrong (parse error, missing field, upload failure, etc.).

In [9]:
ingestor_check = Ingestor(name=collection_name)
files_metadata = ingestor_check.get_file_metadata(return_type="json")
files_metadata

Collection json_multi_file_collection_async initialized for use.


{'file_metadata_count': 25,
 'page': 1,
 'page_size': 25,
 'file_metadata_list': [{'collection_name': 'json_multi_file_collection_async',
   'object_name': '"vsdemo03"."ingest_table_3ccd8b0cc3814328b9945b332b0b4799"',
   'file_name': 'alg-geom9409005.json',
   'md5_hash': '4efd437ebc88675ecee7b42d086d5c9e',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-06-02 11:32:51.561938+00:00'},
  {'collection_name': 'json_multi_file_collection_async',
   'object_name': '"vsdemo03"."ingest_table_3ccd8b0cc3814328b9945b332b0b4799"',
   'file_name': 'alg-geom9509002.json',
   'md5_hash': 'b0be6bc6d10b00051ca40f9a105ac2bb',
   'status': 'ready',
   'error_message': None,
   'last_updated': '2026-06-02 11:32:51.561938+00:00'},
  {'collection_name': 'json_multi_file_collection_async',
   'object_name': '"vsdemo03"."ingest_table_3ccd8b0cc3814328b9945b332b0b4799"',
   'file_name': 'alg-geom9601014.json',
   'md5_hash': 'cfd3fc011a2fd0ed0b72c1d72d43e184',
   'status': 'ready',
   

---
## <b style='font-size:22px;font-family:Arial'>5 · Explore Ingested Data</b>

`get_indexes_embeddings()` returns the full index table — useful for verifying what was ingested, checking field values, and spotting anomalies.

In [11]:
coll = Collection(name=collection_name)
df = coll.get_indexes_embeddings()
df.head(1)

Collection json_multi_file_collection_async initialized for use.


DataBaseName,TableName,TD_ID,td_filename,languages,text,vector_index,vector_index_normalized
vsdemo03,ingest_table_3ccd8b0cc3814328b9945b332b0b4799,1794,alg-geom9412022.json,"[""eng""]","Rf∗x vanishes above dimension l+n. In the notation of t–structure truncations, if x ∈ D(qc/X)≤l, then Rf∗x ∈ D(qc/Y )≤l+n. Pick any x ∈ D(qc/X)≤l, and y ∈ D(qc/Y )≥l+n+1. Then","0.03460693359375,0.033935546875,0.04241943359375,-0.00756072998046875,0.0185089111328125,0.0103759765625,-0.0214996337890625,0.0208282470703125,-0.07000732421875,-0.0086669921875,-0.0142364501953125,0.0167083740234375,-0.052947998046875,0.0275115966796875,0.031097412109375,0.00835418701171875,-0.00456619262695312,-0.050201416015625,0.03271484375,0.08966064453125,-0.000925540924072266,-0.041717529296875,0.0697021484375,-0.0225372314453125,0.0196380615234375,-0.010772705078125,0.031097412109375,0.034423828125,0.06658935546875,-0.021728515625,-0.0066070556640625,-0.0278167724609375,-0.007415771484375,-0.037841796875,-0.0197906494140625,-0.00281524658203125,0.051055908203125,0.0195159912109375,-0.0183868408203125,0.03924560546875,-0.0051727294921875,-0.0182342529296875,-0.0672607421875,-0.01800537109375,0.0254058837890625,0.0247955322265625,-0.02850341796875,-0.02783203125,-0.000240087509155273,0.02813720703125,0.00255012512207031,0.000707626342773438,-0.0628662109375,0.036590576171875,-0.06329345703125,-0.002208","0.0345987342298031,0.0339275076985359,0.0424093827605247,-0.00755893858149648,0.0185045264661312,0.0103735188022256,-0.0214945394545794,0.0208233129233122,-0.0699907392263412,-0.00866493862122297,-0.0142330778762698,0.0167044159024954,-0.0529354549944401,0.0275050792843103,0.0310900453478098,0.00835220795124769,-0.00456511089578271,-0.050189521163702,0.0327070951461792,0.0896394029259682,-0.000925321655813605,-0.0417076461017132,0.0696856379508972,-0.0225318912416697,0.0196334086358547,-0.0107701532542706,0.0310900453478098,0.0344156734645367,0.0665735825896263,-0.0217233672738075,-0.00660549057647586,-0.0278101824223995,-0.00741401454433799,-0.0378328301012516,-0.0197859611362219,-0.00281457952223718,0.0510438121855259,0.0195113681256771,-0.0183824840933084,0.039236307144165,-0.00517150387167931,-0.0182299334555864,-0.0672448053956032,-0.0180011056363583,0.0253998655825853,0.0247896574437618,-0.028496665880084,-0.0278254374861717,-0.000240030625718646,0.0281305406242609,0.00254952092655003,0.0007074587047100"


---
## <b style='font-size:22px;font-family:Arial'>6 · Similarity Search</b>

Returns the top-K most semantically similar records to a natural language question.

In [12]:
coll.similarity_search(
    question="What are the main topics discussed in these papers?",
    top_k=3,
    embedding_model=embedding_model,
)

similar_objects_count:3
similar_objects:
      score DataBaseName                                      TableName  TD_ID           td_filename languages                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    text  index_label
0  0.069604     vsdemo03  ingest_table_3ccd8b0cc3814328b9945b332b0b4799   2667  alg-geom9307008.json   ["eng"]  K  √ − 2  1  1  [LJ,∂∗] = ∇  ∇  = ∇  −  ∇  .  Corollary 4.1 The following is also true:  [LJ,δ∗] = √  [LJ,  ̄δ∗] =  −  √  1  ̄δ  − 1 δ.  −  Proposition 4.3 a). The operators ∂∗ and ∂j anticommute, i. e., ∂∗∂j + ∂j∂∗ = 0

---
## <b style='font-size:22px;font-family:Arial'>7 · Ask a Question (RAG)</b>

`ask()` combines similarity search with an LLM to generate a grounded natural language answer from your collection.  
Configure a `chat_model` (TeradataAI) and the collection will retrieve relevant context, then pass it to the LLM.

In [13]:
# To view a list of hosted embedding and chat models use list_available_models
CollectionManager.list_available_models()

embedding_models:
                        model_id                      model_name provider  status
0        cohere.embed-english-v3                   Embed English   Cohere  ACTIVE
1   cohere.embed-multilingual-v3              Embed Multilingual   Cohere  ACTIVE
2    amazon.titan-embed-image-v1  Titan Multimodal Embeddings G1   Amazon  ACTIVE
3     amazon.titan-embed-text-v1      Titan Embeddings G1 - Text   Amazon  ACTIVE
4   amazon.titan-embed-text-v2:0        Titan Text Embeddings V2   Amazon  ACTIVE
5  amazon.titan-embed-g1-text-02        Titan Text Embeddings v2   Amazon  ACTIVE

chat_models:
                           model_id                         model_name     provider  status
0      nvidia.nemotron-super-3-120b  NVIDIA Nemotron 3 Super 120B A12B       NVIDIA  ACTIVE
1   mistral.mistral-large-2407-v1:0              Mistral Large (24.07)   Mistral AI  ACTIVE
2                         zai.glm-5                              GLM 5         Z.AI  ACTIVE
3  mistral.mistral-7b-inst

In [ ]:
chat_model = TeradataAI(
    api_type="td_hosted",
    model_name="openai.gpt-oss-120b-1:0",
)

answer = coll.ask(
    question="What are the main topics discussed in these papers?",
    embedding_model=embedding_model,
    chat_model=chat_model,
    top_k=5
)
print(answer)

---
## <b style='font-size:22px;font-family:Arial'>8 · Destroy Collection</b>

Clean up the collection from Teradata Vantage when it is no longer needed.

In [15]:
coll.destroy()

Collection destroy request is accepted and in progress
